# 第22章 好みを教える：RLHFとDPO

「どちらの答えが良いか」だけで、モデルの性格を変える

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 好みのデータを見る


In [ ]:
!pip install -q trl peft
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_prefs.jsonl
import json
prefs = [json.loads(l) for l in open("bongo_prefs.jsonl", encoding="utf-8")]
print(len(prefs), "組")
p = prefs[11]
print("質問：", p["question"])
print("良い答え：", p["chosen"])
print("悪い答え：", p["rejected"])

## 報酬モデルと、好みの確率


In [ ]:
import math
for x in [-4, -2, 0, 2, 4]:
    print(f"σ({x:+d}) = {1 / (1 + math.exp(-x)):.3f}")

## DPO：報酬モデルなしで、好みを直接学ぶ


In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=0.1):
    """各引数は、その答え全体の対数確率（トークンごとの log 確率の合計）"""
    margin = beta * ((pi_c - ref_c) - (pi_r - ref_r))
    return -F.logsigmoid(margin)

ref_c, ref_r = torch.tensor(-60.0), torch.tensor(-25.0)     # 学習前：悪い答え（短い）のほうが書きやすい
print("学習前（方策＝参照）：", dpo_loss(ref_c, ref_r, ref_c, ref_r).item())
print("良い答えが +10 書きやすく：", dpo_loss(ref_c + 10, ref_r, ref_c, ref_r).item())
print("悪い答えが -10 書きにくく：", dpo_loss(ref_c, ref_r - 10, ref_c, ref_r).item())
print("両方：", dpo_loss(ref_c + 10, ref_r - 10, ref_c, ref_r).item())

## モデルの対数確率を計算する


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(name)
ref_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device).eval()

@torch.no_grad()
def answer_logprob(model, question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + a]).to(device)
    logprobs = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
    return logprobs[len(p) - 1:].gather(1, ids[0, len(p):, None]).sum().item()

p = prefs[11]
lc, lr = answer_logprob(ref_model, p["question"], p["chosen"]), answer_logprob(ref_model, p["question"], p["rejected"])
print(f"良い答えの対数確率 {lc:.1f}  悪い答えの対数確率 {lr:.1f}")

## TRLのDPOTrainerで学習する


In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

ds = Dataset.from_list([{"prompt": [{"role": "user", "content": p["question"]}],
                         "chosen": [{"role": "assistant", "content": p["chosen"]}],
                         "rejected": [{"role": "assistant", "content": p["rejected"]}]} for p in prefs])
policy = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
config = DPOConfig(output_dir="dpo-bongo", beta=0.1, learning_rate=5e-5, num_train_epochs=3,
                   per_device_train_batch_size=4, logging_steps=1, report_to="none", save_strategy="no",
                   bf16=False, fp16=torch.cuda.is_available())
trainer = DPOTrainer(model=policy, args=config, train_dataset=ds, processing_class=tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
trainer.train()

In [ ]:
for h in trainer.state.log_history:
    if "rewards/margins" in h and h["step"] % 4 == 1:     # 4ステップごとに表示
        print(f"ステップ {h['step']:2d}：損失 {h['loss']:.3f}  良い {h['rewards/chosen']:5.2f}  "
              f"悪い {h['rewards/rejected']:5.2f}  差 {h['rewards/margins']:5.2f}  正解率 {h['rewards/accuracies']:.2f}")

In [ ]:
trainer.model.eval()
for name_, m in [("学習前", ref_model), ("DPO後", trainer.model)]:
    lc = answer_logprob(m, p["question"], p["chosen"])
    lr = answer_logprob(m, p["question"], p["rejected"])
    print(f"{name_}：良い答え {lc:7.1f}  悪い答え {lr:7.1f}")